<div style="font-family:Times New Roman;font-size:14pt;color:black"><b>Лабораторна робота 7</b><br>Рекомендаційні системи<br>Виконав Шиндир Олексій<br>4 курс, 10 група</div>

<a href="https://colab.research.google.com/github/raysewww-lang/mashynne-navchannia/blob/main/%D0%A8%D0%B8%D0%BD%D0%B4%D0%B8%D1%80_%D0%BF%D1%80_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Відкрити в Colab"/></a>

In [1]:
# Підготовка бібліотек і даних. Дані автоматично завантажуються з GitHub; резервний спосіб — data.zip.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
np.random.seed(25)
plt.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','DejaVu Serif'],'font.size':14,'figure.figsize':(9,5)})
pd.set_option('display.max_columns',40)

def data_path(name):
    for folder in [Path('data'),Path('.'),Path('/content/data'),Path('/content')]:
        p=folder/name
        if p.exists():
            return p
    import zipfile, io
    from urllib.request import urlopen
    try:
        url='https://raw.githubusercontent.com/raysewww-lang/mashynne-navchannia/main/data.zip'
        with urlopen(url, timeout=60) as response:
            content=response.read()
        Path('data').mkdir(exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            for member in z.namelist():
                if not member.endswith('/'):
                    (Path('data')/Path(member).name).write_bytes(z.read(member))
        if (Path('data')/name).exists():
            return Path('data')/name
    except Exception as error:
        print('Автоматичне завантаження недоступне:',error)
    from google.colab import files
    import zipfile, io
    print('Завантажте data.zip або файл',name)
    uploaded=files.upload()
    Path('data').mkdir(exist_ok=True)
    for fname,content in uploaded.items():
        if fname.endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                for member in z.namelist():
                    if not member.endswith('/'):
                        (Path('data')/Path(member).name).write_bytes(z.read(member))
        else:
            (Path('data')/Path(fname).name).write_bytes(content)
    p=Path('data')/name
    if not p.exists():
        raise FileNotFoundError(name)
    return p

In [2]:
# Підготовка бібліотеки Surprise. Установка потрібна тільки за відсутності бібліотеки.
import importlib.util,sys,subprocess
if importlib.util.find_spec('surprise') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','scikit-surprise==1.1.5'])
from surprise import Dataset,Reader,SVD,NMF,AlgoBase,accuracy
from surprise.model_selection import GridSearchCV as SurpriseGrid
from surprise.model_selection import KFold as SurpriseKFold,train_test_split as surprise_split

In [3]:
# Завдання 1. Пункти 1–2: завантажити MovieLens 100k бібліотекою Surprise і показати 5 рядків.
ratings=pd.read_csv(data_path('u.data'),sep='	',names=['user','item','rating','timestamp'])
movies=pd.read_csv(data_path('u.item'),sep='|',encoding='latin-1',header=None,usecols=[0,1],names=['item','title'])
display(ratings.head());print(ratings.shape)
print('MovieLens 100k: https://grouplens.org/datasets/movielens/100k/')

reader=Reader(line_format='user item rating timestamp',sep='	',rating_scale=(1,5))
data=Dataset.load_from_file(str(data_path('u.data')),reader=reader)

,user,item,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


(100000, 4)
MovieLens 100k: https://grouplens.org/datasets/movielens/100k/


In [4]:
# Завдання 1. Пункти 3–5: SVD і NMF, підбір параметрів крос-валідацією, вибір за MAE.
folds=SurpriseKFold(n_splits=3,shuffle=True,random_state=25)
settings=[('SVD',SVD,{'n_factors':[20,40],'n_epochs':[20],'reg_all':[0.02,0.1],'random_state':[25]}),('NMF',NMF,{'n_factors':[15,30],'n_epochs':[30],'random_state':[25]})]
searches={};rows=[]
for name,algorithm,params in settings:
    search=SurpriseGrid(algorithm,params,measures=['mae','rmse'],cv=folds,n_jobs=1)
    search.fit(data);searches[name]=search
    rows.append([name,search.best_score['mae'],search.best_params['mae']])
results=pd.DataFrame(rows,columns=['Алгоритм','CV MAE','Параметри']);display(results)
best_name=results.loc[results['CV MAE'].idxmin(),'Алгоритм']
best=searches[best_name].best_estimator['mae']
best.fit(data.build_full_trainset())
print('Найкращий за MAE:',best_name)

,Алгоритм,CV MAE,Параметри
0,SVD,0.743725,"{'n_factors': 40, 'n_epochs': 20, 'reg_all': 0..."
1,NMF,0.763227,"{'n_factors': 15, 'n_epochs': 30, 'random_stat..."


Найкращий за MAE: SVD


In [5]:
# Завдання 1. Пункт 6: 10 рекомендацій для користувача 25 серед фільмів, які він не оцінював.
user_id=25
seen=set(ratings.loc[ratings.user==user_id,'item'])
recs=[]
for item in movies.item:
    if item not in seen:
        recs.append([item,best.predict(str(user_id),str(item)).est])
top=pd.DataFrame(recs,columns=['item','Прогноз']).nlargest(10,'Прогноз').merge(movies,on='item')
display(top[['item','title','Прогноз']])

,item,title,Прогноз
0,318,Schindler's List (1993),4.833185
1,64,"Shawshank Redemption, The (1994)",4.657628
2,515,"Boot, Das (1981)",4.640758
3,483,Casablanca (1942),4.574553
4,313,Titanic (1997),4.551307
5,316,As Good As It Gets (1997),4.502835
6,170,Cinema Paradiso (1988),4.484141
7,603,Rear Window (1954),4.465661
8,251,Shall We Dance? (1996),4.456406
9,178,12 Angry Men (1957),4.432743


In [6]:
# Завдання 2. Власна рекомендаційна система з базовими зміщеннями, її оцінка на відкладених даних і рекомендації.
class MyRecommender(AlgoBase):
    def __init__(self,reg=10,steps=10):
        super().__init__();self.reg=reg;self.steps=steps
    def fit(self,trainset):
        super().fit(trainset)
        self.bu=np.zeros(trainset.n_users);self.bi=np.zeros(trainset.n_items)
        for _ in range(self.steps):
            for u in trainset.all_users():
                self.bu[u]=sum(r-trainset.global_mean-self.bi[i] for i,r in trainset.ur[u])/(self.reg+len(trainset.ur[u]))
            for i in trainset.all_items():
                self.bi[i]=sum(r-trainset.global_mean-self.bu[u] for u,r in trainset.ir[i])/(self.reg+len(trainset.ir[i]))
        return self
    def estimate(self,u,i):
        value=self.trainset.global_mean
        if self.trainset.knows_user(u):value+=self.bu[u]
        if self.trainset.knows_item(i):value+=self.bi[i]
        return value
trainset,testset=surprise_split(data,test_size=0.2,random_state=25)
own=MyRecommender().fit(trainset)
predictions=own.test(testset)
accuracy.mae(predictions);accuracy.rmse(predictions)
selected=searches[best_name].best_estimator['mae'].fit(trainset)
print('Для порівняння, обраний алгоритм на тому самому holdout:')
accuracy.mae(selected.test(testset));accuracy.rmse(selected.test(testset))
own.fit(data.build_full_trainset())
recs=[[i,own.predict(str(user_id),str(i)).est] for i in movies.item if i not in seen]
display(pd.DataFrame(recs,columns=['item','Прогноз']).nlargest(10,'Прогноз').merge(movies,on='item'))

MAE:  0.7454
RMSE: 0.9413


Для порівняння, обраний алгоритм на тому самому holdout:
MAE:  0.7348
RMSE: 0.9327


,item,Прогноз,title
0,318,4.620617,Schindler's List (1993)
1,483,4.613827,Casablanca (1942)
2,64,4.602199,"Shawshank Redemption, The (1994)"
3,12,4.536099,"Usual Suspects, The (1995)"
4,603,4.520954,Rear Window (1954)
5,178,4.477154,12 Angry Men (1957)
6,272,4.433305,Good Will Hunting (1997)
7,513,4.414327,"Third Man, The (1949)"
8,511,4.408172,Lawrence of Arabia (1962)
9,285,4.402192,Secrets & Lies (1996)


In [7]:
# Висновок. SVD і NMF порівняно за MAE у трьох фолдах; найкращий алгоритм наведено в таблиці. Власна система оцінює рейтинг як загальне середнє плюс регуляризовані зміщення користувача та фільму. Якість перевірено на відкладених оцінках; рекомендації виключають уже переглянуті фільми.